# Module 03 — Mô phỏng bộ giải mã / bộ mã hoá / bộ dồn kênh (Decoder / Encoder / Multiplexer)
**AE2.021 · USTH**

🇻🇳 Notebook mô phỏng bảng chân trị của 3 mạch MSI phổ biến nhất trong module: bộ giải mã 3-sang-8, bộ mã
hoá ưu tiên, và bộ dồn kênh 4-sang-1 / 8-sang-1 — đồng thời kiểm chứng công thức n = log₂(N).

🇬🇧 This notebook simulates the truth table of the three most common MSI circuits in this module: a 3-to-8
decoder, a priority encoder, and 4-to-1 / 8-to-1 multiplexers — and verifies the n = log₂(N) formula.

## 1. Bộ giải mã 3-sang-8 (3-to-8 decoder)
🎯 **Phương pháp này trả lời câu hỏi gì?** Với 3 đường địa chỉ, làm sao xác định chính xác đường ra nào
được kích hoạt (=1) cho MỖI tổ hợp địa chỉ, không nhầm lẫn thứ tự bit.

In [1]:
import itertools

def decoder_3to8(a2, a1, a0):
    addr = (a2 << 2) | (a1 << 1) | a0
    outputs = [0]*8
    outputs[addr] = 1
    return outputs

print("A2 A1 A0 | Y0 Y1 Y2 Y3 Y4 Y5 Y6 Y7")
for a2, a1, a0 in itertools.product([0,1], repeat=3):
    outs = decoder_3to8(a2, a1, a0)
    print(f" {a2}  {a1}  {a0}  | " + " ".join(str(o) for o in outs))

A2 A1 A0 | Y0 Y1 Y2 Y3 Y4 Y5 Y6 Y7
 0  0  0  | 1 0 0 0 0 0 0 0
 0  0  1  | 0 1 0 0 0 0 0 0
 0  1  0  | 0 0 1 0 0 0 0 0
 0  1  1  | 0 0 0 1 0 0 0 0
 1  0  0  | 0 0 0 0 1 0 0 0
 1  0  1  | 0 0 0 0 0 1 0 0
 1  1  0  | 0 0 0 0 0 0 1 0
 1  1  1  | 0 0 0 0 0 0 0 1


#### 📤 Đầu ra thật
Đúng như kỳ vọng: ở mỗi hàng, CHÍNH XÁC MỘT ngõ ra bằng 1 (ngõ ra có chỉ số bằng giá trị nhị phân của địa
chỉ A2A1A0), 7 ngõ ra còn lại đều bằng 0 — đây là đặc trưng cốt lõi của một bộ giải mã.

## 2. Bộ dồn kênh (multiplexer) và công thức n = log₂(N)
🎯 **Phương pháp này trả lời câu hỏi gì?** Có đúng N kênh dữ liệu, cần tối thiểu bao nhiêu đường chọn — và
xác nhận công thức n=log₂(N) khớp với các câu hỏi trên trang web (mux 4-sang-1 cần 2 đường chọn, 8-sang-1
cần 3 đường chọn).

In [2]:
import math

def mux(data, select_bits):
    idx = int("".join(str(b) for b in select_bits), 2)
    return data[idx]

# 4-to-1 mux, can 2 duong chon
data4 = [0,1,0,1]
for s1, s0 in itertools.product([0,1], repeat=2):
    print(f"S1={s1} S0={s0} -> Y = data[{int(f'{s1}{s0}',2)}] = {mux(data4,[s1,s0])}")

for N in (2,4,8,16,32):
    n = math.log2(N)
    print(f"N={N:2d} kenh -> can n = log2({N}) = {n:.0f} duong chon")

S1=0 S0=0 -> Y = data[0] = 0
S1=0 S0=1 -> Y = data[1] = 1
S1=1 S0=0 -> Y = data[2] = 0
S1=1 S0=1 -> Y = data[3] = 1
N= 2 kenh -> can n = log2(2) = 1 duong chon
N= 4 kenh -> can n = log2(4) = 2 duong chon
N= 8 kenh -> can n = log2(8) = 3 duong chon
N=16 kenh -> can n = log2(16) = 4 duong chon
N=32 kenh -> can n = log2(32) = 5 duong chon


#### 📤 Đầu ra thật
Bảng xác nhận: N=4 → n=2 đường chọn, N=8 → n=3 đường chọn — khớp đúng với câu hỏi gốc "A four-to-one
multiplexer has two select inputs" (Tooley Ch.9 Q5) và câu tự sinh về mux 8-sang-1.

## 3. Bộ mã hoá ưu tiên đơn giản (priority encoder, 8 đầu vào)
🎯 **Phương pháp này trả lời câu hỏi gì?** Khi NHIỀU đầu vào cùng tích cực một lúc, bộ mã hoá ưu tiên chọn
đầu vào có chỉ số CAO NHẤT để mã hoá — khác với bộ mã hoá thường (chỉ đúng khi có đúng 1 đầu vào tích cực).

In [3]:
def priority_encoder_8to3(inputs):
    for i in range(7, -1, -1):
        if inputs[i] == 1:
            return format(i, '03b')
    return None

tests = [
    [0,0,0,0,0,0,1,0],  # chi input 6 tich cuc
    [0,0,1,0,0,1,0,1],  # nhieu input tich cuc: 2, 5, 7 -> uu tien 7
]
for t in tests:
    print(f"input={t} -> ma hoa (uu tien cao nhat) = {priority_encoder_8to3(t)}")

input=[0, 0, 0, 0, 0, 0, 1, 0] -> ma hoa (uu tien cao nhat) = 110
input=[0, 0, 1, 0, 0, 1, 0, 1] -> ma hoa (uu tien cao nhat) = 111


#### 📤 Đầu ra thật
Khi cả input 2, 5 và 7 đều tích cực, bộ mã hoá ưu tiên chọn đúng input 7 (chỉ số cao nhất) để mã hoá thành
`111`, minh hoạ đúng lý do vì sao encoder cần thêm logic "ưu tiên" so với mã hoá đơn giản.